# 🦾 SmolVLA GPU Server (Colab)

**Serves SmolVLA (450M) for VLA evaluation via Flask + ngrok.**

The backend sends observations (images + joint state + instruction) and receives action predictions (6 joint targets).

### Setup
1. Set runtime to **GPU** (T4 is fine — SmolVLA is only 450M params)
2. Run all cells in order
3. Copy the ngrok URL and set `VLA_WORKER_URL` in your backend `.env`

In [ ]:
#@title 1. Install Dependencies
!pip install -q "lerobot[smolvla]" flask pyngrok pillow

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
#@title 2. Load SmolVLA Model
import torch
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors

MODEL_ID = "lerobot/smolvla_base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_ID} → {DEVICE}...")
policy = SmolVLAPolicy.from_pretrained(MODEL_ID).to(DEVICE).eval()

preprocess, postprocess = make_pre_post_processors(
    policy.config,
    MODEL_ID,
    preprocessor_overrides={"device_processor": {"device": str(DEVICE)}},
)

# Print model info
n_params = sum(p.numel() for p in policy.parameters()) / 1e6
print(f"✅ SmolVLA loaded: {n_params:.0f}M params on {DEVICE}")
print(f"   Config keys: {list(policy.config.__dict__.keys())[:15]}")

# Discover expected input format
cfg = policy.config
print(f"\n   Action dim: {getattr(cfg, 'action_dim', '?')}")
print(f"   Chunk size: {getattr(cfg, 'chunk_size', '?')}")
print(f"   N action steps: {getattr(cfg, 'n_action_steps', '?')}")
print(f"   Image keys: {getattr(cfg, 'image_keys', getattr(cfg, 'input_image_keys', '?'))}")
print(f"   State dim: {getattr(cfg, 'state_dim', getattr(cfg, 'input_state_dim', '?'))}")

In [ ]:
#@title 3. Flask Server Definition
import base64
import io
import json
import time
import traceback
import numpy as np
from PIL import Image
from flask import Flask, request, jsonify

app = Flask(__name__)

# ── Discover model's expected image keys ─────────────────────────────

_image_keys = []
_state_key = "observation.state"
try:
    for key, feat in policy.config.input_features.items():
        if hasattr(feat, 'type') and 'VISUAL' in str(feat.type):
            _image_keys.append(key)
    print(f"  Model image keys: {_image_keys}")
except Exception as e:
    _image_keys = ["observation.images.camera1", "observation.images.camera2"]
    print(f"  Fallback image keys: {_image_keys} ({e})")

# ── Helpers ──────────────────────────────────────────────────────────

def decode_image(b64_str: str) -> Image.Image:
    """Decode a base64 (or data-URI) string → PIL Image."""
    if "," in b64_str:
        b64_str = b64_str.split(",", 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_str))).convert("RGB")


def pil_to_tensor(pil_img: Image.Image) -> torch.Tensor:
    """PIL Image → [C, H, W] float32 tensor in [0, 1]."""
    arr = np.array(pil_img).astype(np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1)  # HWC → CHW


def build_frame(images: dict, state: list, instruction: str, step: int = 0) -> dict:
    """
    Build a LeRobot-compatible observation frame.

    images:      {"top": base64, "side": base64}
    state:       [j1, j2, j3, j4, j5, j6]
    instruction: "Pick up the red cube"
    """
    frame = {}

    # Convert input images to tensors
    img_tensors = []
    for cam_name, b64 in images.items():
        pil = decode_image(b64).resize((256, 256))
        img_tensors.append(pil_to_tensor(pil))

    # Assign to model's expected keys (camera1, camera2, etc.)
    for i, key in enumerate(_image_keys):
        if i < len(img_tensors):
            frame[key] = img_tensors[i]
        else:
            frame[key] = img_tensors[-1] if img_tensors else torch.zeros(3, 256, 256)

    # State (proprioception)
    frame[_state_key] = torch.tensor(state, dtype=torch.float32)

    # Task instruction
    frame["task"] = instruction

    # Required metadata
    frame["timestamp"] = torch.tensor([step * 0.1], dtype=torch.float32)
    frame["frame_index"] = torch.tensor([step], dtype=torch.int64)
    frame["episode_index"] = torch.tensor([0], dtype=torch.int64)
    frame["index"] = torch.tensor([step], dtype=torch.int64)

    return frame


# ── Endpoints ────────────────────────────────────────────────────────

@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "model": MODEL_ID,
        "type": "vla",
        "device": str(DEVICE),
        "params_m": n_params,
        "image_keys": _image_keys,
    })


@app.route("/vla_infer", methods=["POST"])
def vla_infer():
    """
    VLA inference endpoint.

    Input:  {"images": {"top": b64, "side": b64}, "state": [...], "instruction": "...", "step": 0}
    Output: {"action": [j1..j6], "raw_action": [...], "inference_ms": ...}
    """
    try:
        data = request.json
        images = data.get("images", {})
        state = data.get("state", [0.0] * 6)
        instruction = data.get("instruction", "")
        step = data.get("step", 0)

        t0 = time.time()

        # Build frame → preprocess → infer → postprocess
        frame = build_frame(images, state, instruction, step)
        batch = preprocess(frame)

        with torch.inference_mode():
            raw_action = policy.select_action(batch)

        action = postprocess(raw_action)

        # Convert to list
        if isinstance(action, torch.Tensor):
            action_list = action.cpu().numpy().tolist()
        elif isinstance(action, np.ndarray):
            action_list = action.tolist()
        else:
            action_list = list(action)

        # Handle action chunks (nested list)
        if isinstance(action_list, list) and action_list and isinstance(action_list[0], list):
            first_action = action_list[0]
        else:
            first_action = action_list

        elapsed_ms = (time.time() - t0) * 1000
        print(f"  🦾 VLA: {elapsed_ms:.0f}ms → {[f'{a:.3f}' for a in first_action[:6]]}")

        return jsonify({
            "action": first_action[:6],
            "raw_action": action_list,
            "inference_ms": round(elapsed_ms, 1),
        })

    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500


print("✅ Flask app ready — /health and /vla_infer")

In [ ]:
#@title 4. Start Server (ngrok + Flask)
#@markdown Enter your ngrok auth token below:
NGROK_TOKEN = ""  #@param {type:"string"}

import threading
from pyngrok import ngrok
from werkzeug.serving import make_server

# Shutdown previous server if re-running this cell
if '_vla_server' in dir():
    try:
        _vla_server.shutdown()
        print("  Stopped previous server")
    except: pass

# Kill any existing ngrok tunnels
ngrok.kill()

PORT = 5001

# Auth
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
else:
    print("⚠️  No ngrok token — set NGROK_TOKEN above or run: ngrok authtoken <token>")

# Start ngrok tunnel
public_url = ngrok.connect(PORT, bind_tls=True).public_url

print(f"\n{'='*60}")
print(f"🦾 SmolVLA Server Ready!")
print(f"{'='*60}")
print(f"  Public URL: {public_url}")
print(f"  Health:     {public_url}/health")
print(f"  Inference:  {public_url}/vla_infer")
print(f"{'='*60}")
print(f"\n📋 Set this in your backend .env:")
print(f"   VLA_WORKER_URL={public_url}")
print(f"{'='*60}\n")

# Create server with SO_REUSEADDR so re-runs don't fail
import socket
_vla_server = make_server("0.0.0.0", PORT, app)
_vla_server.socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

def run():
    _vla_server.serve_forever()

thread = threading.Thread(target=run, daemon=True)
thread.start()
print("✅ Server running — keep this notebook open!")

In [ ]:
#@title 5. Test Endpoint (Optional)
import requests, base64
from PIL import Image
import io

# Create a dummy test image (gray 256x256)
test_img = Image.new("RGB", (256, 256), (128, 128, 128))
buf = io.BytesIO()
test_img.save(buf, format="JPEG")
b64 = "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

# Health check
try:
    r = requests.get(f"{public_url}/health", headers={"ngrok-skip-browser-warning": "true"}, timeout=10)
    print(f"Health: {r.json()}")
except Exception as e:
    print(f"Health check failed: {e}")

# Inference test
try:
    r = requests.post(
        f"{public_url}/vla_infer",
        json={
            "images": {"top": b64, "side": b64},
            "state": [0.0, -0.8, 1.0, -0.4, 0.0, 0.5],
            "instruction": "Pick up the red cube",
            "step": 0,
        },
        headers={"ngrok-skip-browser-warning": "true"},
        timeout=30,
    )
    result = r.json()
    if "error" in result:
        print(f"❌ Inference error: {result['error']}")
        print(f"   Traceback: {result.get('traceback', '')[:500]}")
    else:
        print(f"✅ Inference OK: action={result['action']}")
        print(f"   Time: {result['inference_ms']}ms")
        print(f"   Raw shape: {len(result.get('raw_action', []))}")
except Exception as e:
    print(f"❌ Inference test failed: {e}")